# Introduction to Strands Agents with MCP

This notebook introduces the core pattern: a **Strands Agent** connected to a **Neo4j MCP Server**. The agent discovers the server's tools at startup and decides which to call — no custom tool wrappers needed.

This is the **Text2Cypher** approach: you ask a question in natural language, the agent retrieves the graph schema, writes a Cypher query, executes it, and interprets the results.

**Learning Objectives:**
- Connect a Strands Agent to an MCP server
- Let the agent discover tools automatically via `list_tools_sync()`
- Query a knowledge graph using natural language (the agent writes its own Cypher)
- Understand the Text2Cypher pattern: schema discovery, query generation, execution

**How it works:**
1. `MCPClient` connects to the Neo4j MCP Server and discovers available tools (`get-schema`, `read-cypher`)
2. The discovered tools are passed directly to the Strands `Agent`
3. When you ask a question, the agent decides which tools to call, in what order
4. Typically: fetch the schema first, then write and execute a Cypher query

**Prerequisites:**
- `../CONFIG.txt` configured with `MCP_GATEWAY_URL` and `MCP_ACCESS_TOKEN`

> The database and indexes are already set up via the pre-deployed MCP server. You do not need to load data yourself.

In [ ]:
%pip install strands-agents strands-agents-tools mcp httpx -q

## 1. Configuration and Imports

In [ ]:
import os

from dotenv import load_dotenv
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient

load_dotenv("../CONFIG.txt")

MODEL_ID = os.getenv("MODEL_ID")
REGION = os.getenv("REGION", "us-east-1")
MCP_GATEWAY_URL = os.getenv("MCP_GATEWAY_URL")
MCP_ACCESS_TOKEN = os.getenv("MCP_ACCESS_TOKEN")

assert MCP_GATEWAY_URL, "MCP_GATEWAY_URL not configured in CONFIG.txt"
assert MCP_ACCESS_TOKEN, "MCP_ACCESS_TOKEN not configured in CONFIG.txt"

print(f"Model:     {MODEL_ID}")
print(f"Region:    {REGION}")
print("\nConfiguration loaded!")

## 2. Initialize Model and MCP Connection

- **BedrockModel**: Configures Claude on Amazon Bedrock with temperature 0 for deterministic responses
- **MCPClient**: Connects to the Neo4j MCP Server and discovers available tools. `list_tools_sync()` returns tool wrappers that the agent can call directly — this is the standard Strands pattern for MCP integration.

In [ ]:
from botocore.config import Config as BotocoreConfig

bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    temperature=0,
    boto_client_config=BotocoreConfig(read_timeout=300),
)

# Open a persistent MCP connection for the notebook session
mcp_client = MCPClient(lambda: streamablehttp_client(
    url=MCP_GATEWAY_URL,
    headers={"Authorization": f"Bearer {MCP_ACCESS_TOKEN}"},
))
mcp_client.__enter__()

# Discover available tools from the MCP server
mcp_tools = mcp_client.list_tools_sync()
tool_names = [t.tool_name for t in mcp_tools]
print(f"MCP tools discovered: {tool_names}")
print(f"\nModel: {MODEL_ID}")

## 3. Create the Agent

Pass the discovered MCP tools directly to the agent. The agent inspects each tool's name, description, and parameter schema to decide when and how to call it — no `@tool` wrappers or manual tool selection needed.

The system prompt instructs the agent to follow a **schema-first approach**: always retrieve the database schema before writing Cypher. This grounds the agent's queries in the actual graph structure.

In [ ]:
SYSTEM_PROMPT = """You are a financial analysis assistant with access to a Neo4j knowledge graph containing SEC 10-K filing data (companies, products, risk factors, asset managers, and document chunks).

Rules:
1. Always retrieve the database schema before writing Cypher queries.
2. Only use read-only Cypher (MATCH, RETURN, WITH, WHERE, ORDER BY, LIMIT).
3. Use modern Cypher syntax:
   - Use elementId(n) instead of id(n)
   - Use COUNT{ pattern } instead of size((pattern))
   - Use EXISTS{ pattern } instead of exists((pattern))
   - Always use $parameter syntax for dynamic values
4. Keep results concise — limit to 10 rows unless asked otherwise.
"""

agent = Agent(
    model=bedrock_model,
    system_prompt=SYSTEM_PROMPT,
    tools=mcp_tools,
)


def query(question: str):
    """Ask the agent a question about the knowledge graph."""
    print(f'Question: "{question}"')
    print("-" * 60)
    response = agent(question)
    print(f"\n{response}")
    return response


print("Agent ready!")

## 4. Query the Knowledge Graph

Ask questions in natural language. Watch the tool calls in the output — the agent will typically call `get-schema` first to learn the graph structure, then write and execute Cypher via `read-cypher`.

In [ ]:
# Schema discovery — the agent calls get-schema and summarizes what it finds
result = query("What is the database schema? Give me a brief summary.")

In [ ]:
# Single-hop traversal: Company → Product
result = query("What products does NVIDIA offer?")

In [ ]:
# Aggregation across the graph
result = query("Which risk factors are shared by more than one company? Show the risk factor and which companies face it.")

In [ ]:
# Multi-relationship query: competition and partnership
result = query("Who are NVIDIA's competitors? Are any of them also partners?")

In [ ]:
# Portfolio analysis: Asset Manager → Company → Risk
result = query("What companies does Berkshire Hathaway own, and what risk factors do those companies face?")

## Summary

You connected a Strands Agent to a Neo4j MCP Server and queried a knowledge graph using natural language:

| Component | Role |
|-----------|------|
| **Strands Agent** | Receives your question, decides which tools to call, writes Cypher, interprets results |
| **MCPClient** | Discovers tools from the MCP server and provides them to the agent |
| **Neo4j MCP Server** | Exposes `get-schema` and `read-cypher` tools over MCP |
| **BedrockModel** | Powers the agent's reasoning with Claude on Amazon Bedrock |

This pattern works well for **structured graph queries** where the agent writes Cypher from scratch. But what about **semantic search** — finding text passages by meaning rather than exact keywords?

Vector search requires generating a numerical embedding (a 1024-dimensional float array) from the query text. If the agent called the MCP server directly, that embedding would flow through the LLM's context window — wasting tokens on numbers the model cannot interpret. The next notebook solves this with a **`@tool` wrapper** that keeps the embedding on the data plane.

---

**Next:** [Vector Search](01_vector_search_mcp.ipynb) — encapsulating embeddings inside a custom tool wrapper